# Example 11.4.3 / 1: Comparing two starts

Companion notebook for **11.4.3 Local and Global Nonlinear Solvers** in the AMPL lab Documentation, section "Comparing two starts". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Comparing two starts

The objective $(x^2-1)^2+0.1x$ is nonconvex. The script deliberately solves from two starts and preserves both results; it does not claim the better of two local solutions is globally optimal.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -2, <= 2;
minimize Wells: (x^2 - 1)^2 + 0.1 * x;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.var["x"].set_value(-1.5)

In [5]:
# run.py
results = []
for start in (-1.5, 1.5):
    ampl.var["x"].set_value(start)
    ampl.solve(solver="ipopt", verbose=False)
    results.append((start, ampl.solve_result,
                    round(ampl.var["x"].value(), 6),
                    round(ampl.obj["Wells"].value(), 6)))
print(results)

[(-1.5, 'solved', -1.012273, -0.100617), (1.5, 'solved', 0.987257, 0.099367)]


### Pure AMPL: data.dat and model.run

The same experiment in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
var x := -1.5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver ipopt;
option ipopt_options 'outlev=0';
option solver_msg 0;
for {start in {-1.5, 1.5}} {
    let x := start;
    solve;
    printf "start=%g result=%s x=%.6f objective=%.6f\n",
        start, solve_result, x, Wells;
}

Ipopt 3.12.13: outlev=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************


suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
start=-1.5 result=solved x=-1.012273 objective=-0.100617
Ipopt 3.12.13: outlev=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

start=1.5 result=solved x=0.987257 objective=0.099367
